# Cleanup, variable selection, and QC

In this notebook we will go through the QC procedure to select, clean, and transform the NHANES data

## Loading data

In [1]:
import os
import pandas as pd
import clarite as cl
import numpy as np

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f347cb260d0>, origin='/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])


In [2]:
#### SET PATHS
os.chdir('..')
mainpath = os.getcwd()
datapath = os.path.join(mainpath, 'Data')

In [3]:
os.chdir(os.path.join(datapath, 'nh_99-06'))
nhanes = pd.read_csv('MainTable.csv').rename(columns={'SEQN':'ID'}).set_index('ID')
var_description = pd.read_csv('VarDescription.csv')\
                     .drop_duplicates()\
                     .set_index('var')
# Convert variable descriptions to a dictionary for convenience
var_descr_dict = var_description['var_desc'].to_dict()

In [4]:
os.chdir(datapath)
survey_design = pd.read_csv("weights_8yr.csv", sep=",")\
                            .rename(columns={'SEQN':'ID'})\
                            .set_index("ID")
survey_design.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT8YR,WTMEC8YR,WTDR8YR,WTSBA8YR,WTSVOC8Y,WTSHM8YR,WTSTH8YR,WTSPP8YR,WTSPO8YR,WTSCY8YR,WTSAU8YR,WTSAF8YR,WTSCI8YR,WTSPH8YR
ID,,,,,,,,,,,,,,,,,
1,1,1,5,2145.745089,2228.103297,3033.064332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,1,7101.667999,7668.099858,7460.967146,NaN,NaN,NaN,23409.459484,NaN,25553.046925,8200.841014,NaN,16536.633786,NaN,NaN
3,1,2,7,10061.881753,10629.233313,7933.223820,NaN,NaN,NaN,NaN,23451.073383,NaN,NaN,NaN,26217.112736,NaN,34581.882658
4,1,1,2,2291.066154,2281.194534,1609.049955,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,2,8,22080.934100,22992.983916,29486.805566,31077.341064,NaN,69045.753325,NaN,NaN,NaN,92261.634160,NaN,49234.403246,NaN,NaN


In [5]:
nhanes.head()

,RIDAGEYR,female,male,black,mexican,other_hispanic,other_eth,SES_LEVEL,education,WTMEC2YR,...,MORTSTAT,MORTSOURCE_NDI,MORTSOURCE_SSA,PERMTH_INT,PERMTH_EXM,CAUSEAVL,UCODE,DIABETES,HYPERTEN,HIPFRACT
ID,,,,,,,,,,,,,,,,,,,,,
1,2,1,0,1,0,0,0,0.0,NaN,10982.898896,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,77,0,1,0,0,0,0,2.0,2.0,28325.384898,...,0.0,NaN,NaN,91.0,90.0,NaN,NaN,NaN,NaN,NaN
3,10,1,0,0,0,0,0,0.0,0.0,46192.256945,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,0,1,1,0,0,0,0.0,NaN,10251.260020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,49,0,1,0,0,0,0,2.0,2.0,99445.065735,...,0.0,NaN,NaN,74.0,74.0,NaN,NaN,NaN,NaN,NaN


In [6]:
var_description.head()

,tab_name,tab_desc,series,module,var_desc,analyzable,binary_ref_group,comment_var,version_date,is_comment,...,mexican_pct,black_N,black_pct,other_hispanic_N,other_hispanic_pct,other_N,other_pct,series_num,category,sub_category
var,,,,,,,,,,,,,,,,,,,,,
LBXHBC,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis B core antibody,1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBDHBG,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis B surface antigen,1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBDHCV,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis C antibody (confirmed),1,2.0,NaN,2009-11-13,0.0,...,0.271826,7269.0,0.244361,1198.0,0.040273,1153.0,0.038760,3,viral infection,NaN
LBDHD,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis D (anti-HDV),1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBXHBS,l02hbs_c,Hepatitis B Surface Antibody,2003-2004,laboratory,Hepatitis B Surface Antibody,1,2.0,NaN,2009-11-13,0.0,...,0.273572,7879.0,0.247472,1317.0,0.041366,1272.0,0.039952,3,viral infection,NaN


In [7]:
os.chdir(datapath)
# These files map variables to their correct weights, and were compiled by reading throught the NHANES codebook
var_weights = pd.read_csv("VarWeights8yr.csv")
var_weights.head()

,variable_name,weight
0,99999,WTMEC8YR
1,ACETAMINOPHEN__CODEINE,WTMEC8YR
2,ACETAMINOPHEN__CODEINE_PHOSPHATE,WTMEC8YR
3,ACETAMINOPHEN__HYDROCODONE,WTMEC8YR
4,ACETAMINOPHEN__HYDROCODONE_BITARTRATE,WTMEC8YR


In [8]:
# Convert the data to two dictionaries for convenience
weights = var_weights.set_index('variable_name')['weight'].to_dict()

## Define phenotypes and covariates


In [9]:
phenotype = ["LBXSCRINV","URXUCR","LBXSCR","LBXSATSI","LBXSAL","URXUMASI","URXUMA","LBXSAPSI","LBXSASSI","LBXSC3SI",
             "LBXSBU","LBXBAP","LBXCPSI","LBXCRP","LBXSCLSI","LBXSCH","LBDHDL","LBDLDL","LBXFER","LBXSGTSI","LBXSGB",
             "LBXGLU","LBXGH","LBXHCY","LBXSIR","LBXSLDSI","LBXMMA","LBXSOSSI","LBXSPH","LBXSKSI","LBXEPP",	
             "LBXSNASI","LBXTIB","LBXSTB","LBXSCA","LBXSTP","LBDPCT","LBXSTR","LBXSUA","LBDBANO","LBXBAPCT",
             "LBDEONO","LBXEOPCT","LBXHCT","LBXHGB","LBDLYMNO","LBXMCHSI","LBXLYPCT","LBXMCVSI","LBXMPSI","LBDMONO",
             "LBXMOPCT","LBXPLTSI","LBXRBCSI","LBXRDW","LBDNENO","LBXNEPCT","LBXIRN"]
for v in phenotype:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
        
covariates = ["black", "mexican", "other_hispanic", "other_eth", "SES_LEVEL", "RIDAGEYR", "SDDSRVYR","BMXBMI"]

	LBXSCRINV = 1/Creatinine (mg/dL)
	URXUCR = Creatinine, urine (mg/dL)
	LBXSCR = Creatinine (mg/dL)
	LBXSATSI = ALT (U/L)
	LBXSAL = Albumin (g/dL)
	URXUMASI = Albumin, urine (mg/L) SI
	URXUMA = Albumin, urine (ug/mL)
	LBXSAPSI = Alkaline phosphotase (U/L)
	LBXSASSI = AST (U/L)
	LBXSC3SI = Bicarbonate (mmol/L)
	LBXSBU = Blood urea nitrogen (mg/dL)
	LBXBAP = Bone alkaline phosphotase (ug/L)
	LBXCPSI = C-peptide: SI(nmol/L)
	LBXCRP = C-reactive protein(mg/dL)
	LBXSCLSI = Chloride (mmol/L)
	LBXSCH = Cholesterol, total (mg/dL)
	LBDHDL = Direct HDL-Cholesterol (mg/dL)
	LBDLDL = LDL-cholesterol (mg/dL)
	LBXFER = Ferritin (ng/mL)
	LBXSGTSI = GGT (U/L)
	LBXSGB = Globulin (g/dL)
	LBXGLU = Glucose, plasma (mg/dL)
	LBXGH = Glycohemoglobin (%)
	LBXHCY = Homocysteine (umol/L)
	LBXSIR = Iron (ug/dL)
	LBXSLDSI = LDH (U/L)
	LBXMMA = Methylmalonic acid (umol/L)
	LBXSOSSI = Osmolality (mOsml/L)
	LBXSPH = Phosphorus (mg/dL)
	LBXSKSI = Potassium (mmol/L)
	LBXEPP = Protoporphyrin (ug/dL RBC)
	LBXSNASI = Sodi

## Keep only adults

In [10]:
keep_adults = nhanes['RIDAGEYR'] >= 18
nhanes      = nhanes.loc[keep_adults]
nhanes.head()

,RIDAGEYR,female,male,black,mexican,other_hispanic,other_eth,SES_LEVEL,education,WTMEC2YR,...,MORTSTAT,MORTSOURCE_NDI,MORTSOURCE_SSA,PERMTH_INT,PERMTH_EXM,CAUSEAVL,UCODE,DIABETES,HYPERTEN,HIPFRACT
ID,,,,,,,,,,,,,,,,,,,,,
2,77,0,1,0,0,0,0,2.0,2.0,28325.384898,...,0.0,NaN,NaN,91.0,90.0,NaN,NaN,NaN,NaN,NaN
5,49,0,1,0,0,0,0,2.0,2.0,99445.065735,...,0.0,NaN,NaN,74.0,74.0,NaN,NaN,NaN,NaN,NaN
6,19,1,0,0,0,0,1,0.0,2.0,39656.600444,...,0.0,NaN,NaN,87.0,86.0,NaN,NaN,NaN,NaN,NaN
7,59,1,0,1,0,0,0,NaN,0.0,25525.423409,...,0.0,NaN,NaN,77.0,76.0,NaN,NaN,NaN,NaN,NaN
10,43,0,1,1,0,0,0,NaN,1.0,22445.808572,...,0.0,NaN,NaN,79.0,79.0,NaN,NaN,NaN,NaN,NaN


In [11]:
cl.describe.summarize(nhanes)

22,624 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Remove participants with missing values in covariates

In [12]:
nhanes = cl.modify.rowfilter_incomplete_obs(nhanes, only=covariates)

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 3,687 of 22,624 observations (16.30%) due to NA values in any of 8 variables


In [13]:
cl.describe.summarize(nhanes)

18,937 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Remove variables without weights

In [14]:
keep  = set(weights.keys()) | set(phenotype + covariates)
nhanes = nhanes[[c for c in list(nhanes) if c in keep]]

In [15]:
cl.describe.summarize(nhanes)

18,937 observations of 1,085 variables
	0 Binary Variables
	0 Categorical Variables
	1,084 Continuous Variables
	1 Unknown-Type Variables



## Variable selection and clenup

Here we will:
- Drop any variables that are indeterminant according to the NHANES data dictionary
- Drop variables that don't have 4 year weights
- Drop any non-environmental exposures (examples physical fitness)
- Determine covariates and phenotypes
- From the remainder of the variables split them based on their type
- Manually inspect the definition of ambiguous variables and determine their type
- Merge binary into categorical variables


In [16]:
#### RECODING VARIABLES

# SMQ077 and DDB100 have Refused/Don't Know for "7" and "9" values.
# We will recode them as nan
nhanes = cl.modify.recode_values(nhanes, {7: np.nan, 9: np.nan}, only=['SMQ077', 'DBD100'])

Running recode_values
--------------------------------------------------------------------------------
Replaced 11 values from 18,937 observations in 2 variables


In [17]:
#### DROPING PHYSICAL FITNESS

# Droping physical fitness measurements, because they are not proxies for environmental exposures
phys_fitness_vars = ["CVDVOMAX","CVDESVO2","CVDS1HR","CVDS1SY","CVDS1DI","CVDS2HR","CVDS2SY","CVDS2DI","CVDR1HR","CVDR1SY","CVDR1DI","CVDR2HR","CVDR2SY","CVDR2DI","physical_activity"]
for v in phys_fitness_vars:
    print(f"\t{v} = {var_descr_dict[v]}")
nhanes = nhanes.drop(columns=phys_fitness_vars)

#Nikki also droped this unknown pesticide ('URXP08')

	CVDVOMAX = Predicted VO2max (ml/kg/min)
	CVDESVO2 = Estimated VO2max (ml/kg/min)
	CVDS1HR = Stage 1 heart rate (per min)
	CVDS1SY = Stage 1 systolic BP (mm Hg)
	CVDS1DI = Stage 1 diastolic BP (mm Hg)
	CVDS2HR = Stage 2 heart rate (per min)
	CVDS2SY = Stage 2 systolic BP (mm Hg)
	CVDS2DI = Stage 2 diastolic BP (mm Hg)
	CVDR1HR = Recovery 1 heart rate (per min)
	CVDR1SY = Recovery 1 systolic BP (mm Hg)
	CVDR1DI = Recovery 1 diastolic BP (mm Hg)
	CVDR2HR = Recovery 2 heart rate (per min)
	CVDR2SY = Recovery 2 systolic BP (mm Hg)
	CVDR2DI = Recovery 2 diastolic BP (mm Hg)
	physical_activity = Physical Activity (MET-based rank)


In [18]:
#### DROPING INDETERMINATE VARIABLES

indeterminent_vars = ["house_type","hepa","hepb","house_age","current_past_smoking","house_age","DUQ130","DMDMARTL","income"]
print("Removing variables with indeterminate meanings:")
for v in indeterminent_vars:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
nhanes = nhanes.drop(columns=indeterminent_vars)

Removing variables with indeterminate meanings:
	house_type = house type
	hepa = hepatitis a
	hepb = hepatitis b
	house_age = house age
	current_past_smoking = Current or Past Cigarette Smoker?
	house_age = house age
	DUQ130 = #days used needle for street drugs/year


In [19]:
cl.describe.summarize(nhanes)

18,937 observations of 1,062 variables
	0 Binary Variables
	0 Categorical Variables
	1,061 Continuous Variables
	1 Unknown-Type Variables



## Adjust lipids if on statins

In [20]:
#Adjust for statins (LDL=LDL/0.7, TC=TC/0.8) (2)
statins   = ["ATORVASTATIN_CALCIUM", "SIMVASTATIN", "PRAVASTATIN_SODIUM", "FLUVASTATIN_SODIUM"]
for v in statins:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
nhanes.loc[(nhanes[statins].sum(axis=1) > 0), 'LBDLDL'] = nhanes.loc[(nhanes[statins].sum(axis=1) > 0), 'LBDLDL']/0.7

	ATORVASTATIN_CALCIUM = ATORVASTATIN_CALCIUM
	SIMVASTATIN = SIMVASTATIN
	PRAVASTATIN_SODIUM = PRAVASTATIN_SODIUM
	FLUVASTATIN_SODIUM = FLUVASTATIN_SODIUM


## Separate males and females

In [21]:
keep_females = nhanes.female == 1
keep_males   = nhanes.male   == 1

In [22]:
nhanes_females = nhanes.loc[keep_females]
nhanes_males   = nhanes.loc[keep_males]

In [23]:
# Drop the male, female column, and the white as well
nhanes_females = nhanes_females.drop(['female', 'male','white'], axis=1)
nhanes_males   = nhanes_males.drop(['female', 'male','white'], axis=1)

## QC

We will remove variables that have less than 200 non-nan values

In [24]:
nhanes_females = cl.modify.colfilter_min_n(nhanes_females, skip=phenotype + covariates)
nhanes_males   = cl.modify.colfilter_min_n(nhanes_males, skip=phenotype + covariates)

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 0 of 0 binary variables
Testing 0 of 0 categorical variables
Testing 993 of 1,059 continuous variables
	Removed 99 (9.97%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 0 of 0 binary variables
Testing 0 of 0 categorical variables
Testing 993 of 1,059 continuous variables
	Removed 111 (11.18%) tested continuous variables which had less than 200 non-null values.


## Categorize variables

In [25]:
nhanes_females = cl.modify.categorize(nhanes_females)
nhanes_males   = cl.modify.categorize(nhanes_males)

Running categorize
--------------------------------------------------------------------------------
37 of 961 variables (3.85%) are classified as constant (1 unique value).
364 of 961 variables (37.88%) are classified as binary (2 unique values).
38 of 961 variables (3.95%) are classified as categorical (3 to 6 unique values).
486 of 961 variables (50.57%) are classified as continuous (>= 15 unique values).
0 of 961 variables (0.00%) were dropped.
36 of 961 variables (3.75%) were not categorized and need to be set manually.
	35 variables had between 6 and 15 unique values
	1 variables had >= 15 values but couldn't be converted to continuous (numeric) values
Running categorize
--------------------------------------------------------------------------------
59 of 949 variables (6.22%) are classified as constant (1 unique value).
350 of 949 variables (36.88%) are classified as binary (2 unique values).
26 of 949 variables (2.74%) are classified as categorical (3 to 6 unique values).
492 o

## Remove constant variables

In [26]:
var_types_f    = cl.describe.get_types(nhanes_females)
var_constant_f = var_types_f[var_types_f == 'constant'].index

var_types_m    = cl.describe.get_types(nhanes_males)
var_constant_m = var_types_m[var_types_m == 'constant'].index

var_constant   = (var_constant_f.append(var_constant_m)).unique()

for v in list(var_constant):
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
    
nhanes_females = nhanes_females.drop(columns=var_constant_f)
nhanes_males   = nhanes_males.drop(columns=var_constant_m)

	ANTIBIOTIC_UNSPECIFIED = ANTIBIOTIC_UNSPECIFIED
	industry_manufacturing = industry_manufacturing
	industry_other = industry_other
	occupation_household = occupation_household
	occupation_midmanage = occupation_midmanage
	SMD100MN = Menthol indicator
	SMD100FL = Filter type
	DRD350K = Refused on shellfish eaten past 30 days
	DRD370V = Refused on fish eaten past 30 days
	OMEGA_6_FATTY_ACIDS_NA = OMEGA_6_FATTY_ACIDS_NA
	OMEGA_9_FATTY_ACIDS_NA = OMEGA_9_FATTY_ACIDS_NA
	OTHER_FATTY_ACIDS_NA = OTHER_FATTY_ACIDS_NA
	OTHER_OMEGA_3_FATTY_ACIDS_NA = OTHER_OMEGA_3_FATTY_ACIDS_NA
	CALCIUM_PPM = CALCIUM_PPM
	L_GLUTAMINE_gm = L_GLUTAMINE_gm
	GLYCINE_gm = GLYCINE_gm
	MAGNESIUM_PPM = MAGNESIUM_PPM
	ALPHA_TOCOPHEROL_NA = ALPHA_TOCOPHEROL_NA
	blood_cancer_self_report = blood_cancer_self_report
	gallbladder_cancer_self_report = gallbladder_cancer_self_report
	larynx_cancer_self_report = larynx_cancer_self_report
	nervous_cancer_self_report = nervous_cancer_self_report
	pancreatic_cancer_self_report = pa

## Examine unknown variables

In both datasets the variable L_GLUTAMINE_gm is incorrectly assigned as categorical, but it is continuous

In [28]:
v = "L_GLUTAMINE_gm"
print(f"\t{v} = {var_descr_dict[v]}\n")
#nhanes_females = cl.modify.make_continuous(nhanes_females, only=[v]) #There is no such variables
nhanes_males   = cl.modify.make_continuous(nhanes_males, only=[v])

	L_GLUTAMINE_gm = L_GLUTAMINE_gm

Running make_continuous
--------------------------------------------------------------------------------
Set 1 of 890 variable(s) as continuous, each with 9,090 observations


In [38]:
var_types_f   = cl.describe.get_types(nhanes_females)
var_unknown_f = var_types_f[var_types_f == 'unknown'].index

var_types_m   = cl.describe.get_types(nhanes_males)
var_unknown_m = var_types_m[var_types_m == 'unknown'].index

print('Unknown variables in females:')
for v in list(var_unknown_f):
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
        
print('\nUnknown variables in males:')
for v in list(var_unknown_m):
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")

Unknown variables in females:
	LBDBANO = Basophils number
	URXPPX = 2-isopropoxyphenol (ug/L)
	DRD350AQ = # of times clams eaten in past 30 days
	DRD350BQ = # of times crabs eaten in past 30 days
	DRD350DQ = # of times lobsters eaten past 30 days
	DRD350FQ = # of times oysters eaten in past 30 days
	DRD350GQ = # of times scallops eaten past 30 days
	DRD370AQ = # of times breaded fish products eaten
	DRD370DQ = # of times catfish eaten in past 30 days
	DRD370EQ = # of times cod eaten in past 30 days
	DRD370FQ = # of times flatfish eaten past 30 days
	DRD370KQ = # of times pollock eaten in past 30 days
	DRD370NQ = # of times sardines eaten past 30 days
	DRD370RQ = # of times trout eaten in past 30 days
	DRD370UQ = # of times other unknown fish eaten
	ALANINE_mg = ALANINE_mg
	ARGININE_mg = ARGININE_mg
	BETA_CAROTENE_mg = BETA_CAROTENE_mg
	CAFFEINE_mg = CAFFEINE_mg
	HISTIDINE_mg = HISTIDINE_mg
	ISOLEUCINE_mg = ISOLEUCINE_mg
	LEUCINE_mg = LEUCINE_mg
	LYSINE_mg = LYSINE_mg
	PHENYLALANINE_mg 

In both datasets all unknown variables are continuous, except area which is categorical

In [40]:
nhanes_females = cl.modify.make_continuous(nhanes_females, only=var_unknown_f[:-1])
nhanes_females = cl.modify.make_categorical(nhanes_females, only=var_unknown_f[-1])

nhanes_males = cl.modify.make_continuous(nhanes_males, only=var_unknown_m[:-1])
nhanes_males = cl.modify.make_categorical(nhanes_males, only=var_unknown_m[-1])

Running make_continuous
--------------------------------------------------------------------------------
Set 35 of 924 variable(s) as continuous, each with 9,847 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 1 of 924 variable(s) as categorical, each with 9,847 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 21 of 890 variable(s) as continuous, each with 9,090 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 1 of 890 variable(s) as categorical, each with 9,090 observations


In [41]:
cl.describe.summarize(nhanes_females)

9,847 observations of 924 variables
	364 Binary Variables
	39 Categorical Variables
	521 Continuous Variables
	0 Unknown-Type Variables



In [42]:
cl.describe.summarize(nhanes_males)

9,090 observations of 890 variables
	350 Binary Variables
	26 Categorical Variables
	514 Continuous Variables
	0 Unknown-Type Variables



## Filtering

In [43]:
# 200 non-na samples
nhanes_females = cl.modify.colfilter_min_n(nhanes_females)
nhanes_males   = cl.modify.colfilter_min_n(nhanes_males)

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 364 of 364 binary variables
	Removed 0 (0.00%) tested binary variables which had less than 200 non-null values.
Testing 39 of 39 categorical variables
	Removed 0 (0.00%) tested categorical variables which had less than 200 non-null values.
Testing 521 of 521 continuous variables
	Removed 0 (0.00%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 350 of 350 binary variables
	Removed 0 (0.00%) tested binary variables which had less than 200 non-null values.
Testing 26 of 26 categorical variables
	Removed 0 (0.00%) tested categorical variables which had less than 200 non-null values.
Testing 514 of 514 continuous variables
	Removed 0 (0.00%) tested continuous variables which had less than 200 non-null values.


In [44]:
# 200 samples per category
nhanes_females = cl.modify.colfilter_min_cat_n(nhanes_females, skip=[c for c in covariates + phenotype if c in nhanes_females.columns] )
nhanes_males   = cl.modify.colfilter_min_cat_n(nhanes_males, skip=[c for c in covariates + phenotype if c in nhanes_males.columns] )

Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 360 of 364 binary variables
	Removed 271 (75.28%) tested binary variables which had a category with less than 200 values.
Testing 37 of 39 categorical variables
	Removed 31 (83.78%) tested categorical variables which had a category with less than 200 values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 346 of 350 binary variables
	Removed 272 (78.61%) tested binary variables which had a category with less than 200 values.
Testing 24 of 26 categorical variables
	Removed 16 (66.67%) tested categorical variables which had a category with less than 200 values.


In [45]:
# 90percent zero filter
nhanes_females = cl.modify.colfilter_percent_zero(nhanes_females)
nhanes_males   = cl.modify.colfilter_percent_zero(nhanes_males)

Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 521 of 521 continuous variables
	Removed 26 (4.99%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 514 of 514 continuous variables
	Removed 29 (5.64%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.


In [46]:
cl.describe.summarize(nhanes_females)

9,847 observations of 596 variables
	93 Binary Variables
	8 Categorical Variables
	495 Continuous Variables
	0 Unknown-Type Variables



In [47]:
cl.describe.summarize(nhanes_males)

9,090 observations of 573 variables
	78 Binary Variables
	10 Categorical Variables
	485 Continuous Variables
	0 Unknown-Type Variables



In [49]:
# Take note of which variables were differently typed in each dataset
print("Correcting differences in variable types between discovery and replication")
# Merge current type series
dtypes = pd.DataFrame({'females':cl.describe.get_types(nhanes_females),
                       'males':cl.describe.get_types(nhanes_males)
                       })
diff_dtypes = dtypes.loc[(dtypes['females'] != dtypes['males']) &
                         (~dtypes['females'].isna()) &
                         (~dtypes['males'].isna())]

# There are no differences

Correcting differences in variable types between discovery and replication


## Keeping variables that are in both datasets

In [73]:
both = set(list(nhanes_females)) & set(list(nhanes_males))
nhanes_females = nhanes_females[both]
nhanes_males   = nhanes_males[both]
print(f"{len(both)} variables in common")

555 variables in common


## Inspect variables

In [74]:
def inspect_variables(data, suffix=''):
    #### Plot variables to manually inspect them
    os.chdir(os.path.join(mainpath, 'Results', 'Plots'))
    
    var_types  = cl.describe.get_types(data)
    var_binary = var_types[var_types == 'binary'].index
    var_categorical = var_types[var_types == 'categorical'].index
    var_categorical  = var_types[var_types == 'continuous'].index
    
    vartypes = [var_binary, var_categorical, var_categorical, phenotype, covariates]
    names    = ['var_binary', 'var_categorical', 'var_categorical', 'phenotype', 'covariates']
    
    for i in range(0, len(vartypes)):
        outname = names[i] + suffix + '.pdf'
        cl.plot.distributions(data,
                              filename=outname,
                              continuous_kind='count',
                              nrows=4,
                              ncols=3,
                              quality='medium',
                              variables=vartypes[i])

In [75]:
inspect_variables(nhanes_females, '_f')

Generating a 6 page PDF for 71 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or 

Generating a 40 page PDF for 476 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 40 page PDF for 476 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 5 page PDF for 58 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 1 page PDF for 8 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit ke

In [76]:
inspect_variables(nhanes_males, '_m')

Generating a 6 page PDF for 71 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or 

Generating a 40 page PDF for 476 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 40 page PDF for 476 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 5 page PDF for 58 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 1 page PDF for 8 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit ke

## Summaries

In [77]:
cl.describe.summarize(nhanes_females)

9,847 observations of 555 variables
	71 Binary Variables
	8 Categorical Variables
	476 Continuous Variables
	0 Unknown-Type Variables



In [78]:
cl.describe.summarize(nhanes_males)

9,090 observations of 555 variables
	71 Binary Variables
	8 Categorical Variables
	476 Continuous Variables
	0 Unknown-Type Variables



## Save cleaned data

In [79]:
os.chdir(os.path.join(mainpath, 'Results'))
nhanes_females.to_csv('CleanData_Females.csv')
nhanes_males.to_csv('CleanData_Males.csv')

## Transform variables (not sure if need to be)

We'll log transform the phenotype variables that are non-normal. If they are negative skewed, we'll mirror them by subtracting the maximum value plus 1 and then log transform.

In [32]:
transform = ['LBDHDL','LBDNENO','LBDPCT','URXUCR']
nhanes    = cl.modify.transform(nhanes, transform_method='log', only=transform)

Running transform
--------------------------------------------------------------------------------
Transformed 'LBDNENO' using 'log'
Transformed 'LBDPCT' using 'log'
Transformed 'URXUCR' using 'log'
Transformed 'LBDHDL' using 'log'


In [33]:
cl.plot.distributions(nhanes,
                      filename="var_phenotype_transform_distributions.pdf",
                      continuous_kind='count',
                      nrows=4,
                      ncols=3,
                      quality='medium',
                      variables=phenotype)

Generating a 5 page PDF for 58 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)
